[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/07_Fundamentos_de_inferencia.ipynb)

# Fundamentos de inferencia estadística (opcional)

"*El pensamiento estadístico será algún día tan necesario para la ciudadanía eficiente como la capacidad de leer y escribir" H. G. Wells (1866-1946)*

## De describir a inferir

Hasta este punto del curso hemos usado la estadística para **describir** los datos que tenemos: medidas numéricas, tablas y visualizaciones. Pero en los negocios casi nunca observamos todos los datos que nos interesan: encuestamos a una muestra de clientes, probamos una campaña con un subconjunto de usuarios, auditamos una muestra de facturas. La **inferencia estadística** es el conjunto de métodos que permite llegar a conclusiones sobre una **población** a partir de una **muestra**, cuantificando la incertidumbre de esas conclusiones.

Cuatro conceptos son la base de todo lo que sigue:

| Concepto | Definición | Ejemplo |
|----------|-----------|---------|
| **Población** | Conjunto completo de elementos de interés | Todos los hogares del país |
| **Muestra** | Subconjunto de la población que sí observamos | Los hogares encuestados |
| **Parámetro** | Valor que describe a la población; generalmente es desconocido y se denota con letras griegas: $\mu$ (media), $\sigma$ (desviación estándar), $\pi$ (proporción) | El ingreso promedio de *todos* los hogares |
| **Estadístico** | Valor calculado con la muestra, usado para estimar el parámetro: $\bar{x}$, $s$, $p$ | El ingreso promedio de los hogares *encuestados* |

La pregunta central de la inferencia es: *si hubiéramos tomado otra muestra, ¿habríamos llegado a una conclusión diferente?*

## Variabilidad muestral y el Teorema del Límite Central

Dos muestras distintas de la misma población producen dos promedios distintos. A esa variación se le llama **variabilidad muestral**, y la distribución de los valores que tomaría un estadístico a lo largo de muchas muestras posibles se llama **distribución muestral**.

El **Teorema del Límite Central (TLC)** establece que, si el tamaño de muestra es suficientemente grande (como referencia, $n \geq 30$), el promedio muestral $\bar{x}$:
- sigue una distribución aproximadamente **normal**, *aunque la variable original no sea normal*,
- centrada en la media poblacional $\mu$,
- con una desviación estándar igual a $\sigma/\sqrt{n}$, llamada **error estándar**.

Comprobémoslo con datos reales. El ingreso corriente de los hogares (ENIGH) tiene una distribución marcadamente asimétrica; sin embargo, veremos que los *promedios* de muestras repetidas se comportan de forma aproximadamente normal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/enigh2025.xlsx')
ingreso = df['ing_cor']

# Simulación: promedios de 1000 muestras de tamaño 100
rng = np.random.default_rng(42)
medias = [ingreso.sample(100, random_state=i).mean() for i in range(1000)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(ingreso, bins=50, color='steelblue')
axes[0].set_title('Ingreso de los hogares\n(distribución asimétrica)')
axes[1].hist(medias, bins=30, color='seagreen')
axes[1].axvline(ingreso.mean(), color='red', linestyle='--', label='Media poblacional')
axes[1].set_title('Promedios de 1000 muestras (n=100)\n(aproximadamente normal)')
axes[1].legend()
plt.tight_layout();

Observa el resultado: aunque la distribución del ingreso es fuertemente asimétrica, la distribución de los *promedios muestrales* es aproximadamente normal y está centrada en la media poblacional (línea roja).

La dispersión de la gráfica de la derecha es el **error estándar**: qué tanto varía el promedio de una muestra a otra. Si aumentas el tamaño de muestra $n$, esa distribución se vuelve más angosta, es decir, las estimaciones son más precisas.

**¿Por qué importa?** Prácticamente todas las pruebas estadísticas de este curso descansan en este resultado: conocemos cómo se comporta un estadístico bajo muestreo repetido y eso nos permite juzgar si un resultado observado es *atribuible al azar* o no.

## Intervalos de confianza

Un estadístico puntual como $\bar{x}$ casi nunca coincide exactamente con el parámetro. Un **intervalo de confianza (IC)** comunica un rango plausible para el parámetro:

$$\text{estimación} \pm \text{margen de error}$$

donde el margen de error, para un IC del 95%, es aproximadamente 2 veces el error estándar.

**Interpretación correcta**: si repitiéramos el muestreo muchas veces y construyéramos el intervalo en cada ocasión, aproximadamente el 95% de esos intervalos contendrían el verdadero parámetro. La confianza es una propiedad del *procedimiento*, no de un intervalo en particular.

**Interpretación incorrecta (muy común)**: "hay una probabilidad del 95% de que $\mu$ esté dentro de este intervalo". El parámetro es un valor fijo, no aleatorio; lo que varía de muestra en muestra es el intervalo.

Dos lecturas prácticas para negocios:
- Un IC ancho indica poca precisión: muestra pequeña o mucha variabilidad. Reportar solo el promedio ocultaría esa incertidumbre.
- Si el IC de una *diferencia* (entre dos campañas, dos segmentos) incluye el cero, no hay evidencia clara de que la diferencia exista.

## La lógica de una prueba de hipótesis

Una prueba de hipótesis funciona como un juicio:

| En un juicio | En una prueba estadística |
|--------------|---------------------------|
| El acusado se presume inocente | Se asume la hipótesis nula ($H_0$): *no hay efecto o diferencia* |
| El fiscal presenta evidencia | Los datos de la muestra |
| ¿La evidencia es suficiente para condenar? | ¿Los datos son suficientemente improbables bajo $H_0$? |
| Veredicto: culpable / no culpable | Decisión: rechazar / no rechazar $H_0$ |

El procedimiento general es:
1. Plantear $H_0$ (el status quo: no hay diferencia, no hay relación) y $H_1$ (lo que se desea evidenciar).
2. Calcular un estadístico de prueba con los datos.
3. Obtener el **p-valor**: *la probabilidad de observar un resultado tan extremo como el obtenido, suponiendo que $H_0$ es cierta*.
4. Comparar contra el nivel de significancia $\alpha$ (usualmente 0.05): si $p < \alpha$, se rechaza $H_0$.

Dos precisiones importantes:
- El p-valor **no** es la probabilidad de que $H_0$ sea cierta; mide qué tan compatible es el resultado observado con $H_0$.
- "No rechazar $H_0$" **no** significa "aceptar que $H_0$ es verdadera": puede que simplemente la muestra no tenga suficiente información para detectar el efecto (igual que un veredicto de *no culpable* no demuestra inocencia).

### Errores tipo I y tipo II

Como la decisión se toma con información parcial, siempre hay riesgo de equivocarse, y hay dos formas de hacerlo:

| | $H_0$ es verdadera | $H_0$ es falsa |
|---|---|---|
| **Rechazar $H_0$** | Error tipo I (probabilidad $\alpha$): *falso positivo* | Decisión correcta |
| **No rechazar $H_0$** | Decisión correcta | Error tipo II (probabilidad $\beta$): *falso negativo* |

En términos de negocio: un error tipo I es concluir que una campaña funciona cuando no lo hace (invertir en algo inútil); un error tipo II es descartar una campaña que sí funcionaba (perder una oportunidad). Reducir uno de los dos riesgos tiende a aumentar el otro; el balance depende del costo de cada error para la organización.

## Significancia estadística vs. relevancia práctica

El p-valor depende de dos cosas: del tamaño del efecto **y** del tamaño de la muestra. Esto tiene dos consecuencias que todo analista debe tener presentes:

- **Con muestras muy grandes, casi cualquier diferencia resulta "significativa"**. Por ejemplo, comparar una tasa de conversión de 2.00% contra 2.05% con dos millones de usuarios por grupo puede arrojar un p-valor menor a 0.05. La diferencia *existe*, pero ¿justifica el costo de implementar el cambio? Esa es una decisión de negocio, no estadística.
- **Con muestras pequeñas, un efecto real y grande puede no alcanzar significancia** simplemente porque hay poca información (falta de potencia).

Por eso, un buen reporte de resultados incluye tres elementos, no solo uno:
1. La **magnitud del efecto** en unidades del negocio (pesos, puntos porcentuales, minutos).
2. Su **intervalo de confianza** (la precisión de la estimación).
3. El **p-valor** (la evidencia contra $H_0$).

*Significativo* en estadística solo quiere decir "difícilmente atribuible al azar"; no significa "importante". La relevancia práctica se juzga comparando la magnitud del efecto contra los costos y beneficios de actuar en consecuencia.

## Mapa de las pruebas del curso

Esta tabla resume las pruebas que veremos en los siguientes notebooks. Todas siguen la misma lógica que acabamos de revisar; lo que cambia es el tipo de variables y la pregunta de negocio.

| Pregunta | Tipo de variables | Prueba (y alternativa) | Notebook |
|----------|-------------------|------------------------|----------|
| ¿Los datos siguen una distribución normal? | 1 cuantitativa | Shapiro-Wilk, K-S-Lilliefors | 08 |
| ¿Dos variables cualitativas están asociadas? | 2 cualitativas | Chi cuadrada (exacta de Fisher) | 10 |
| ¿Dos variables cuantitativas están relacionadas? | 2 cuantitativas | Correlación de Pearson (Spearman) | 11 |
| ¿Difiere una proporción entre dos grupos? | 1 cualitativa + 1 proporción | Prueba z de dos proporciones | 14 |
| ¿Difiere un promedio entre dos grupos? | 1 cualitativa (2 grupos) + 1 cuantitativa | Prueba t (Welch, Mann-Whitney) | 14 |
| ¿Difiere un promedio entre tres o más grupos? | 1 cualitativa (3+ grupos) + 1 cuantitativa | ANOVA (Kruskal-Wallis) | 15 |
| ¿Cómo explicar una variable con varias otras? | 1 cuantitativa + varias explicativas | Regresión lineal múltiple | 16-18 |

## Preguntas de autoevaluación

**Pregunta 1**. Un analista compara el ticket promedio de dos sucursales y obtiene un p-valor de 0.03. ¿Cuál es la interpretación correcta?

A) Hay un 3% de probabilidad de que la hipótesis nula sea cierta.
B) Si no existiera diferencia real entre sucursales, un resultado tan extremo como el observado ocurriría en cerca del 3% de las muestras.
C) Hay un 97% de probabilidad de que exista una diferencia entre sucursales.
D) La diferencia entre sucursales es grande.

**Pregunta 2**. En una prueba A/B con dos millones de usuarios por grupo, la nueva versión del sitio aumenta la conversión de 2.00% a 2.05% (p < 0.001). ¿Cuál es la conclusión más adecuada?

A) Implementar el cambio de inmediato: el resultado es altamente significativo.
B) Evaluar si una mejora de 0.05 puntos porcentuales justifica el costo de implementar y mantener el cambio.
C) Repetir la prueba con una muestra más grande.
D) Descartar el resultado: con muestras tan grandes el p-valor no es confiable.

**Pregunta 3**. Se reporta un intervalo de confianza del 95% para el gasto promedio mensual de los clientes: [$1,200, $1,800]. ¿Cuál afirmación es correcta?

A) El 95% de los clientes gasta entre $1,200 y $1,800 al mes.
B) Hay una probabilidad del 95% de que el gasto promedio poblacional esté entre $1,200 y $1,800.
C) Si se repitiera el muestreo muchas veces, cerca del 95% de los intervalos así construidos contendrían el gasto promedio poblacional.
D) El promedio de cualquier muestra futura caerá entre $1,200 y $1,800 el 95% de las veces.

*Respuestas: 1-B, 2-B, 3-C*

## Referencias
- Spiegelhalter, D. (2019). *The Art of Statistics: Learning from Data*. Pelican Books.
- Wheelan, C. (2013). *Naked Statistics: Stripping the Dread from the Data*. W. W. Norton.
- Reinhart, A. (2015). *Statistics Done Wrong: The Woefully Complete Guide*. No Starch Press.